# Wan2.2 VAE Latent Space Image Interpolation

### Overview

This notebook demonstrates how to interpolate between two images using the **frozen VAE** (`WanVideoVAE38`)
from Wan2.2-TI2V-5B in latent space.

### Pipeline
1. Encode two images into latent space via the VAE encoder
2. Interpolate between the two latent vectors using **COG** (Closed-form Optimal Gaussian) correction
3. Decode interpolated latents back to images via the VAE decoder

### The COG Method
Standard linear interpolation (LERP) causes **norm collapse** at midpoints:
$\text{Var}(z_{interp}) = \beta \cdot I$ where $\beta = (1-\alpha)^2 + \alpha^2 < 1$.

COG corrects this with a simple rescaling:
$$z_{COG} = \frac{(1-\alpha) \cdot z_A + \alpha \cdot z_B}{\sqrt{(1-\alpha)^2 + \alpha^2}}$$

This ensures the interpolated latent stays within the VAE's prior distribution $\mathcal{N}(0, I)$,
producing higher-quality decoded images.

### Reference
- **COG**: Bodin et al., "Linear Combinations of Latents in Diffusion Models: Interpolation and Beyond", arXiv 2408.08558, 2024

### 1. Setup & Imports

In [ ]:
# pip install ipykernel matplotlib

import math
import sys
import os

import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Add project root to path
sys.path.insert(0, os.path.abspath(".."))

from src.fastwam.models.wan22.wan_video_vae import WanVideoVAE38
from src.fastwam.models.wan22.helpers.io import ModelConfig, load_state_dict
from src.fastwam.models.wan22.helpers.state_dict_converters import wan_video_vae_state_dict_converter
from src.fastwam.models.wan22.helpers.init import instantiate_module_on_device

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

### 2. Load the Wan2.2 VAE

VAE specifications:
- `z_dim = 48` (latent channels)
- `upsampling_factor = 16` (spatial compression 16x)
- `temporal_downsample_factor = 4` (temporal compression 4x)
- For a single image: T=1 → latent_T=1

In [ ]:
# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.bfloat16

# VAE model config
vae_config = ModelConfig(
    model_id="Wan-AI/Wan2.2-TI2V-5B",
    origin_file_pattern="Wan2.2_VAE.pth",
)
# Redirect to DiffSynth-Studio converted safetensors
vae_config.model_id = "DiffSynth-Studio/Wan-Series-Converted-Safetensors"
vae_config.origin_file_pattern = "Wan2.2_VAE.safetensors"

# Download if needed
vae_config.download_if_necessary()
vae_path = str(vae_config.path)
print(f"VAE checkpoint path: {vae_path}")

# Instantiate and load weights
vae = instantiate_module_on_device(
    WanVideoVAE38,
    {},
    device=DEVICE,
    dtype=TORCH_DTYPE,
)
state_dict = load_state_dict(vae_path, torch_dtype=TORCH_DTYPE, device="cpu")
state_dict = wan_video_vae_state_dict_converter(state_dict)
vae.load_state_dict(state_dict, strict=False)
vae.eval()
vae.requires_grad_(False)
del state_dict

print(f"VAE loaded successfully!")
print(f"  z_dim: {vae.z_dim}")
print(f"  upsampling_factor: {vae.upsampling_factor}")
print(f"  temporal_downsample_factor: {vae.temporal_downsample_factor}")
print(f"  mean (first 8): {vae.mean[:8].tolist()}")
print(f"  std  (first 8): {vae.std[:8].tolist()}")

### 3. Helper Functions

Image loading/preprocessing, VAE encoding/decoding, and COG interpolation.

In [ ]:
def load_and_preprocess(image_path_or_pil, target_size=None):
    """
    Load an image and preprocess it into VAE input format.

    Args:
        image_path_or_pil: Path to image (str) or PIL.Image
        target_size: (H, W) target dimensions. If None, auto-pad to multiples of 16.

    Returns:
        tensor: [1, 3, 1, H, W] normalized to [-1, 1]
        orig_size: (orig_W, orig_H) for reference
    """
    if isinstance(image_path_or_pil, str):
        img = Image.open(image_path_or_pil).convert("RGB")
    else:
        img = image_path_or_pil.convert("RGB")

    orig_size = img.size  # (W, H)

    if target_size is not None:
        img = img.resize((target_size[1], target_size[0]), Image.LANCZOS)
    else:
        # Pad to multiples of 16 (required by VAE with upsampling_factor=16)
        w, h = img.size
        new_h = (h + 15) // 16 * 16
        new_w = (w + 15) // 16 * 16
        if (new_h != h) or (new_w != w):
            print(f"  Resizing: ({h}, {w}) -> ({new_h}, {new_w})")
            img = img.resize((new_w, new_h), Image.LANCZOS)

    # Convert to tensor: [C, H, W] -> [1, C, 1, H, W] (B=1, T=1)
    arr = np.array(img).astype(np.float32) / 127.5 - 1.0  # [0,255] -> [-1,1]
    tensor = torch.from_numpy(arr).permute(2, 0, 1)       # [C, H, W]
    tensor = tensor.unsqueeze(0).unsqueeze(2)              # [1, C, 1, H, W]

    return tensor, orig_size


def encode_image(vae, img_tensor):
    """
    Encode a single image to latent.

    Args:
        vae: WanVideoVAE38 instance
        img_tensor: [1, 3, 1, H, W]

    Returns:
        latent: [1, 48, 1, h, w] normalized latent
    """
    latent = vae.encode([img_tensor[0]], device=img_tensor.device)
    return latent


def decode_latent(vae, latent):
    """
    Decode a latent tensor to PIL Image.

    Args:
        vae: WanVideoVAE38 instance
        latent: [1, 48, 1, h, w] or [48, 1, h, w]

    Returns:
        PIL.Image
    """
    if latent.ndim == 4:
        latent = latent.unsqueeze(0)  # [1, 48, 1, h, w]
    video = vae.decode([latent[0]], device=latent.device)
    video = video.squeeze(0).float().clamp(-1, 1)
    arr = ((video[:, 0].permute(1, 2, 0) + 1.0) * 127.5).to(torch.uint8).cpu().numpy()
    return Image.fromarray(arr)


def cog_interpolate(z_a, z_b, alpha):
    """
    COG (Closed-form Optimal Gaussian) corrected interpolation.

    From: Bodin et al., "Linear Combinations of Latents in Diffusion Models",
    arXiv 2408.08558, 2024.

    For latents drawn from N(0, I), the linear combination
    z_lerp = (1-alpha)*z_a + alpha*z_b has variance
    beta = (1-alpha)^2 + alpha^2 times the original.

    COG divides by sqrt(beta) to restore unit variance,
    keeping the interpolated latent within the VAE prior N(0, I).

    Note: Wan2.2 VAE encoding applies (z - mean) / std normalization,
    so output latents are approximately N(0, I) and the
    sqrt(beta) correction applies directly.
    """
    z_lerp = (1 - alpha) * z_a + alpha * z_b
    beta = (1 - alpha) ** 2 + alpha ** 2
    if beta > 1e-8:
        return z_lerp / math.sqrt(beta)
    return z_lerp


def interpolate_and_decode(vae, z_a, z_b, num_frames):
    """
    Interpolate between two latents using COG and decode all frames.

    Args:
        vae: WanVideoVAE38
        z_a, z_b: Latent tensors [1, 48, 1, h, w]
        num_frames: Number of frames to generate (including endpoints)

    Returns:
        frames: List[PIL.Image]
        alphas: np.ndarray of alpha values
    """
    frames = []
    alphas = np.linspace(0, 1, num_frames)

    for alpha in alphas:
        z_interp = cog_interpolate(z_a, z_b, alpha)
        img = decode_latent(vae, z_interp)
        frames.append(img)

    return frames, alphas


print("Helper functions defined successfully!")

### 4. Theory: Why Does COG Matter?

#### The Norm Collapse Problem

Suppose the VAE latent distribution is $\mathcal{N}(0, I)$. Under linear interpolation:

$$z_{interp} = (1-\alpha) \cdot z_A + \alpha \cdot z_B$$

The variance at intermediate points is:

$$\text{Var}(z_{interp}) = \beta \cdot I, \quad \beta = (1-\alpha)^2 + \alpha^2$$

At $\alpha = 0.5$, $\beta = 0.5$ — variance is **halved**. This pushes the interpolated latent
out of the VAE's training prior $\mathcal{N}(0, I)$, leading to degraded decoded images.

#### COG Solution

$$z_{COG} = \frac{z_{LERP}}{\sqrt{\beta}} = \frac{(1-\alpha) \cdot z_A + \alpha \cdot z_B}{\sqrt{(1-\alpha)^2 + \alpha^2}}$$

This restores $\text{Var}(z_{COG}) = I$ for all $\alpha$.

#### Advantages
- **Mathematically exact**: closed-form solution, no numerical optimization needed
- **As fast as LERP**: only one extra division per interpolation
- **Theoretical guarantee**: interpolated latent stays exactly within $\mathcal{N}(0, I)$

### 5. Load Images

Two structured images (checkerboard and concentric rings) are generated for immediate testing.
Comment out the generator calls and uncomment the `Image.open` block below to use your own images.

In [ ]:
# Generate two structured images for interpolation testing.
# Replace these with your own Image.open(...) calls to use real photos.

def _make_checkerboard(h, w, size=32, color_a=(220, 180, 140), color_b=(80, 120, 180)):
    """Generate a checkerboard pattern."""
    arr = np.zeros((h, w, 3), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            if ((y // size) + (x // size)) % 2 == 0:
                arr[y, x] = color_a
            else:
                arr[y, x] = color_b
    return Image.fromarray(arr)


def _make_rings(h, w, color_bg=(40, 40, 40), color_ring=(220, 180, 60)):
    """Generate concentric rings."""
    arr = np.zeros((h, w, 3), dtype=np.uint8)
    cx, cy = w // 2, h // 2
    for y in range(h):
        for x in range(w):
            d = int(np.sqrt((x - cx) ** 2 + (y - cy) ** 2))
            if (d // 25) % 2 == 0:
                arr[y, x] = color_ring
            else:
                arr[y, x] = color_bg
    return Image.fromarray(arr)


H, W = 256, 256
img_a = _make_checkerboard(H, W, size=32, color_a=(240, 200, 150), color_b=(70, 100, 160))
img_b = _make_rings(H, W, color_bg=(30, 30, 30), color_ring=(200, 160, 50))

# # ---- Or load your own images ----
# img_a = Image.open("path/to/image_a.jpg").convert("RGB")
# img_b = Image.open("path/to/image_b.jpg").convert("RGB")
# # Ensure both images have the same size
# if img_a.size != img_b.size:
#     target_w = min(img_a.width, img_b.width)
#     target_h = min(img_a.height, img_b.height)
#     target_h = (target_h + 15) // 16 * 16
#     target_w = (target_w + 15) // 16 * 16
#     img_a = img_a.resize((target_w, target_h), Image.LANCZOS)
#     img_b = img_b.resize((target_w, target_h), Image.LANCZOS)

print(f"Image A: {img_a.size}, Image B: {img_b.size}")

# Display
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_a)
axes[0].set_title("Image A  (checkerboard)")
axes[0].axis("off")
axes[1].imshow(img_b)
axes[1].set_title("Image B  (rings)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

### 6. Encode Images to Latent Space

In [ ]:
# Preprocess and encode
tensor_a, _ = load_and_preprocess(img_a)
tensor_b, _ = load_and_preprocess(img_b)

z_a = encode_image(vae, tensor_a.to(DEVICE).to(TORCH_DTYPE))
z_b = encode_image(vae, tensor_b.to(DEVICE).to(TORCH_DTYPE))

print(f"Latent shape: {z_a.shape}")
print(f"  B=1, C={z_a.shape[1]} (z_dim), T=1, H={z_a.shape[3]}, W={z_a.shape[4]}")
print(f"  (spatial compression: {vae.upsampling_factor}x)")

# Inspect latent statistics
z_a_flat = z_a.reshape(-1)
z_b_flat = z_b.reshape(-1)
expected_norm = math.sqrt(z_a.numel())
print(f"\nLatent statistics:")
print(f"  z_A: mean={z_a_flat.mean().item():.4f}, std={z_a_flat.std().item():.4f}, norm={z_a_flat.norm().item():.2f}")
print(f"  z_B: mean={z_b_flat.mean().item():.4f}, std={z_b_flat.std().item():.4f}, norm={z_b_flat.norm().item():.2f}")
print(f"  Expected for N(0,1): mean~0, std~1, norm~{expected_norm:.2f}")
print(f"  Cosine similarity: {F.cosine_similarity(z_a_flat.unsqueeze(0), z_b_flat.unsqueeze(0)).item():.4f}")

### 7. Run COG Interpolation and Visualize

In [ ]:
NUM_FRAMES = 9  # Number of interpolation frames (including endpoints)

# Run COG interpolation
frames, alphas = interpolate_and_decode(vae, z_a, z_b, NUM_FRAMES)
print(f"Generated {len(frames)} frames")

# Visualize
fig, axes = plt.subplots(1, NUM_FRAMES, figsize=(NUM_FRAMES * 2.0, 3))
for col, (frame, alpha) in enumerate(zip(frames, alphas)):
    axes[col].imshow(frame)
    label = "Start" if col == 0 else ("End" if col == NUM_FRAMES - 1 else f"α={alpha:.2f}")
    axes[col].set_title(label, fontsize=9)
    axes[col].axis("off")

plt.suptitle("COG Latent Space Interpolation", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 8. Quantitative Analysis: Latent Norm vs α

Verify that COG keeps the interpolated latent within the $\mathcal{N}(0, I)$ prior by maintaining constant norm across all α values.

In [ ]:
alphas_dense = np.linspace(0, 1, 50)

# Compute LERP (uncorrected) for comparison
lerp_norms = []
cog_norms = []
cog_stddevs = []

for alpha in alphas_dense:
    # LERP (uncorrected)
    z_lerp = (1 - alpha) * z_a + alpha * z_b
    lerp_norms.append(z_lerp.norm().item())
    # COG (corrected)
    z_cog = cog_interpolate(z_a, z_b, alpha)
    cog_norms.append(z_cog.norm().item())
    cog_stddevs.append(z_cog.std().item())

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Norm plot
ax1.plot(alphas_dense, lerp_norms, label="LERP (uncorrected)", color="#e74c3c", linewidth=2)
ax1.plot(alphas_dense, cog_norms, label="COG (corrected)", color="#3498db", linewidth=2)
ax1.axhline(y=expected_norm, color='gray', linestyle='--', alpha=0.7, label=f'Expected ~{expected_norm:.0f}')
ax1.set_xlabel('α')
ax1.set_ylabel('L2 Norm')
ax1.set_title('Latent Norm vs α')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Std dev plot
ax2.plot(alphas_dense, cog_stddevs, color="#3498db", linewidth=2, label="COG")
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.7, label='Expected ~1.0')
ax2.set_xlabel('α')
ax2.set_ylabel('Std Dev')
ax2.set_title('Latent Std Dev vs α (COG)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('COG Correction: Norm Preservation', fontsize=14)
plt.tight_layout()
plt.show()

# Print key values
print(f"Expected norm for N(0,1) with {z_a.numel()} dimensions: {expected_norm:.2f}")
mid_idx = len(alphas_dense) // 2
print(f"  LERP at α=0.5: norm={lerp_norms[mid_idx]:.2f} (collapsed to {lerp_norms[mid_idx]/expected_norm*100:.1f}%)")
print(f"  COG  at α=0.5: norm={cog_norms[mid_idx]:.2f}, std={cog_stddevs[mid_idx]:.4f}")

### Summary

- **LERP** (uncorrected) suffers from norm collapse: at α=0.5 the latent norm drops to ~70% of the original, pushing intermediate representations out of the VAE's training distribution.
- **COG** corrects this with a simple `1/sqrt(β)` rescaling that restores unit variance exactly, keeping latents within the VAE prior N(0, I).
- COG is as fast as LERP (only one extra division) and requires no training or numerical optimization.

### Recommendation

Always use COG for Wan2.2 VAE latent interpolation. It guarantees that interpolated latents stay in-distribution, producing the best possible decoded images from the frozen VAE.

### Limitation

For two images with very different content, a straight-line path in latent space may not pass through high-density regions of the data manifold. In such cases, diffusion-based methods (e.g., flow matching with the Video DiT) may produce more natural transitions — this is left for future work.